In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import joblib
import os

In [2]:
print("Loading enriched data...")
model_data = pd.read_parquet("../data/processed/preprocessed_data_enriched.parquet")

model_data["time_step"] = (model_data["date"] - model_data["date"].min()).dt.days

# Split Date (Last 28 days for validation)
max_date = model_data["date"].max()
split_date = max_date - pd.Timedelta(days=28)

# 1. Define Boolean Masks (NO .copy() duplicates!)
train_mask = model_data["date"] < split_date
val_mask = model_data["date"] >= split_date

# 2. Define Features (dropping absolute time markers)
drop_cols = ["id", "d", "date", "sales", "wm_yr_wk", "time_step"]
features = [col for col in model_data.columns if col not in drop_cols]

# 3. Pull X and y slices on the fly using .loc (Saves massive amounts of RAM)
X_train = model_data.loc[train_mask, features]
y_train = model_data.loc[train_mask, "sales"]

X_val = model_data.loc[val_mask, features]
y_val = model_data.loc[val_mask, "sales"]

print(f"Memory-optimized split complete! X_train shape: {X_train.shape}")



Loading enriched data...
Memory-optimized split complete! X_train shape: (4529102, 82)


In [3]:
# 4. Linear Regression (Trend Baseline)
print("Training Linear Regression (Trend Baseline)...")
lr_model = LinearRegression()

# Fit using ONLY the time_step slice directly from model_data
lr_model.fit(
    model_data.loc[train_mask, ["time_step"]], 
    model_data.loc[train_mask, "sales"]
)

# Predict baseline trend directly into standalone arrays
lr_pred_train = lr_model.predict(model_data.loc[train_mask, ["time_step"]])
lr_pred_val = lr_model.predict(model_data.loc[val_mask, ["time_step"]])

# Calculate Residuals (Actual Sales - Trend) without duplicating dataframes
y_train_resid = model_data.loc[train_mask, "sales"] - lr_pred_train
y_val_resid = model_data.loc[val_mask, "sales"] - lr_pred_val

print(f"Memory-optimized split & trend calculation complete! X_train shape: {X_train.shape}")

Training Linear Regression (Trend Baseline)...
Memory-optimized split & trend calculation complete! X_train shape: (4529102, 82)


In [4]:
print("Injecting Trend Meta-Features into LightGBM...")
# We use .assign() to safely add a column to our slice without MemoryErrors
X_train = X_train.assign(lr_trend_baseline=lr_pred_train)
X_val = X_val.assign(lr_trend_baseline=lr_pred_val)

# 5. Training LightGBM on Residuals
print("Training LightGBM on Residuals...")

lgb_model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    n_estimators=1500,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    max_depth=8,
    min_child_weight=50,
    num_leaves=64,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train, y_train_resid,
    eval_set=[(X_train, y_train_resid), (X_val, y_val_resid)],
    eval_names=['train_resid', 'val_resid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=50)
    ]
)

# 6. Predict and Evaluate
print("Calculating Final Hybrid Predictions...")
# Predict the residuals for the Validation set
lgb_resid_pred = lgb_model.predict(X_val)

# Final Prediction = Linear Regression Baseline + LightGBM Residual Prediction
final_hybrid_pred = lr_pred_val + lgb_resid_pred

# Since we can't sell negative items, floor predictions at 0
final_hybrid_pred = np.clip(final_hybrid_pred, a_min=0, a_max=None)

# Performance Metrics
rmse = np.sqrt(mean_squared_error(y_val, final_hybrid_pred))
absolute_errors = np.abs(y_val - final_hybrid_pred)
wmape = np.sum(absolute_errors) / np.sum(y_val)
business_accuracy = (1 - wmape) * 100

print("\n--- Hybrid Model Performance ---")
print(f"Validation RMSE:   {rmse:.4f}")
print(f"WMAPE:             {wmape:.4f} ({wmape * 100:.2f}%)")
print(f"Business Accuracy: {business_accuracy:.2f}%")

# 7. Save Models
os.makedirs("../models", exist_ok=True)
joblib.dump(lr_model, "../models/lr_trend_model.pkl")
joblib.dump(lgb_model, "../models/lgbm_residual_model.pkl")
print("\nSaved both LR and LightGBM models successfully!")

Injecting Trend Meta-Features into LightGBM...
Training LightGBM on Residuals...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.422853 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3876
[LightGBM] [Info] Number of data points in the train set: 4529102, number of used features: 83
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 0.000000
Training until validation scores don't improve for 50 rounds
[50]	train_resid's rmse: 2.5639	val_resid's rmse: 2.12257
[100]	train_resid's rmse: 2.38125	val_resid's rmse: 2.02074
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[150]	train_resid's rm